**Avvio locale e Colab:** preparare la cartella completa seguendo il [README del laboratorio](../README.md). Conservare gli identificativi delle celle e salvare gli output nel proprio runtime.

# C01 — Computer Vision · Lezione 21
## Trustworthy Computer Vision

**Dimostrazione completa su CPU.** Leggiamo insieme una diagnostica di corruzioni controllate, i confronti fra gruppi, la qualità delle probabilità, l’astensione e un benchmark locale. Non sono richiesti nuovi esperimenti da consegnare. Le domande N01–N10 servono alla discussione nello studio e alla clinic dei progetti.

**Avvio locale:** aprire questo notebook in Jupyter o PyCharm. Il file `dati/digits_cv2026.npz` deve trovarsi nella cartella `notebook/`, accanto a `studenti/`. Eseguire in ordine da C03.

**Avvio in Colab:** caricare il notebook con **File → Carica notebook**. Caricare nello stesso runtime lo ZIP dei materiali di questa lezione: impostare `CARICA_ZIP_COLAB = True` in C03, scegliere lo ZIP e poi riportare il valore a `False`. È sufficiente **CPU**; non selezionare una GPU. C03 installa soltanto NumPy, Matplotlib e IPython se mancanti in Colab. Il disco del runtime è temporaneo: C31 prepara un archivio da scaricare. Dopo un riavvio rieseguire C03–C31 e ricaricare i materiali se assenti.

Gli output già presenti sono una **corsa locale CPU**; non sono misure di Colab. I tempi cambiano con ambiente e carico del computer.

## C02 — Percorso e protocollo


Le celle testuali intermedie introducono termini e domande prima dei calcoli. Ogni blocco richiede i precedenti; in caso di perdita dello stato ripartire da C03. Non è necessario rieseguire tutto per leggere gli output salvati.

**N01.** Qual è l’unità valutata e quali dati possiamo usare per stimare le prestazioni in una camera diversa? Prima di rispondere distinguere un’immagine, uno scrivente e un dominio di acquisizione.

In [ ]:
# C03 — Ambiente, cartella dei materiali e salvataggio
import sys, os, json, csv, hashlib, platform, time, zipfile
from pathlib import Path
import importlib.util

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
CARICA_ZIP_COLAB = False
DOWNLOAD_OUTPUTS = False  # attivare in C31 per scaricare l'archivio in Colab
missing = [name for name in ("numpy", "matplotlib", "IPython")
           if importlib.util.find_spec(name) is None]
if missing:
    if not IN_COLAB:
        raise RuntimeError("Installare nell'ambiente del notebook: " + ", ".join(missing))
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

if IN_COLAB and CARICA_ZIP_COLAB:
    from google.colab import files
    uploaded = files.upload()
    dest = Path.cwd()/"materiali_lezione21"
    dest.mkdir(exist_ok=True)
    for name, payload in uploaded.items():
        if not name.lower().endswith(".zip"):
            continue
        import io
        with zipfile.ZipFile(io.BytesIO(payload)) as archive:
            for member in archive.infolist():
                candidate = (dest/member.filename).resolve()
                if not candidate.is_relative_to(dest.resolve()):
                    raise ValueError("Percorso non valido nell'archivio")
            archive.extractall(dest)

bases = [Path.cwd(), *Path.cwd().parents]
candidates = []
for base in bases[:5]:
    candidates.extend([base/"dati/digits_cv2026.npz",
                       base/"notebook/dati/digits_cv2026.npz",
                       base/"materiali_lezione21/notebook/dati/digits_cv2026.npz",
                       base/"materiali_lezione21/dati/digits_cv2026.npz"])
found = [p for p in candidates if p.is_file()]
if not found:
    extra = list((Path.cwd()/"materiali_lezione21").glob("**/dati/digits_cv2026.npz"))
    found.extend(extra)
if not found:
    raise FileNotFoundError("Caricare lo ZIP dei materiali nello stesso runtime e ripetere C03.")
DATA_PATH = found[0].resolve()
MATERIALS = DATA_PATH.parent.parent
OUTPUT = Path.cwd()/"output_lezione21"
OUTPUT.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(OUTPUT/".mplconfig"))
import numpy as np
import matplotlib
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, FileLink
SEED = 20261130
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 115, "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.2})
COLORS = {"teal": "#138C94", "orange": "#C76E28", "blue": "#305C88", "red": "#A63B44"}
def savefig(name):
    plt.savefig(OUTPUT/name, bbox_inches="tight", dpi=180)
    plt.show()
    plt.close()
def table(headers, rows):
    display(Markdown("| " + " | ".join(headers) + " |\n|" + "---|"*len(headers)
                     + "\n" + "\n".join("| " + " | ".join(map(str, row)) + " |" for row in rows)))
print("Ambiente:", "Colab, CPU" if IN_COLAB else "locale, CPU")
print("Python", platform.python_version(), "· NumPy", np.__version__, "· Matplotlib", matplotlib.__version__)
print("Dati:", DATA_PATH.name, "· output:", OUTPUT.name)

## C04 — Dati e baseline prima dello stress test

Digits contiene immagini 8×8 con etichette assegnate 0–9. Riusiamo **1077 immagini di training e 360 di validation già viste nel corso**. Non leggiamo gli indici di test e non valutiamo il test. Il file conserva gli stessi indici delle lezioni precedenti; non disponiamo delle identità degli scriventi e non possiamo dimostrare una separazione per persona.

La baseline è **k-nearest neighbours (kNN)**, con `k = 5`: per una query trova le cinque immagini di training più vicine nella distanza euclidea e vota la classe. Non addestra una rete. Le intensità sono divise per 16; non stimiamo statistiche dai dati da valutare. I pareggi scelgono l’indice di classe minore.

Per una confidenza euristica aggiungiamo `0.1` ai voti di ogni classe e normalizziamo la somma: queste probabilità **non sono calibrate**. Non scegliamo `k`, corruzione o soglie in base ai risultati per sostenere nuove prestazioni di test.

In [ ]:
# C05 — Controlli, split riusato e kNN completo
DATA_SHA256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
assert DATA_SHA256 == "972401a5680cf0a1c5ef60295fb915f4efbd51d7a39b35c0cfa39d73cc33ca1e"
with np.load(DATA_PATH, allow_pickle=False) as data:
    images = data["images"].copy()
    labels = data["target"].astype(int)
    train_idx = data["train_idx"].copy()
    val_idx = data["val_idx"].copy()
assert images.shape == (1797, 8, 8)
assert len(train_idx) == 1077 and len(val_idx) == 360
assert not np.intersect1d(train_idx, val_idx).size
X_ref = images[train_idx].reshape(-1, 64).astype(np.float64)/16.0
y_ref = labels[train_idx]
X_val_images = images[val_idx].astype(np.float64)/16.0
y_val = labels[val_idx]
K_CLASSES, K_NEIGHBORS, VOTE_SMOOTHING = 10, 5, 0.1
ref_norm = np.sum(X_ref**2, axis=1)
def predict_knn(x):
    x = np.asarray(x, dtype=np.float64).reshape(-1, 64)
    d2 = np.maximum(np.sum(x*x, axis=1)[:, None] + ref_norm[None, :] - 2*x@X_ref.T, 0.0)
    # Ordine stabile anche per distanze identiche.
    nearest = np.argsort(d2, axis=1, kind="stable")[:, :K_NEIGHBORS]
    votes = (y_ref[nearest][..., None] == np.arange(K_CLASSES)[None, None, :]).sum(axis=1)
    p = (votes + VOTE_SMOOTHING)/(K_NEIGHBORS + K_CLASSES*VOTE_SMOOTHING)
    return p.argmax(axis=1), p
pred_clean, p_clean = predict_knn(X_val_images)
clean_accuracy = float(np.mean(pred_clean == y_val))
print(f"Training: {len(train_idx)} · validation riusata: {len(val_idx)} · accuracy originale: {clean_accuracy:.6f}")
print("Nessun risultato sul test; nessuna separazione verificata fra scriventi.")

## C06 — Cambiare l’immagine mantenendo il confronto appaiato

Ogni immagine di validation è confrontata con **la propria versione trasformata**. Applichiamo separatamente:

- **Sfocatura:** media 3×3, con estensione dei valori al bordo.
- **Contrasto ridotto:** `x′ = 0.6 x + 0.2`, per intensità nell’intervallo `[0, 1]`.

Le etichette assegnate restano uguali, ma una corruzione può rendere un’immagine ambigua. Questa diagnostica non dimostra che la distribuzione condizionata `p(y | x)` sia invariata, né costituisce un nuovo test su una camera esterna. Nessuna corruzione è scelta per migliorare la baseline.

**N02.** Un calo di accuracy sotto sfocatura prova un problema in qualsiasi dominio futuro? Quale confronto aggiungereste per una camera diversa?

In [ ]:
# C07 — Trasformazioni, misure appaiate e casi che cambiano esito
padded = np.pad(X_val_images, ((0, 0), (1, 1), (1, 1)), mode="edge")
blurred = sum(padded[:, dy:dy+8, dx:dx+8] for dy in range(3) for dx in range(3))/9.0
low_contrast = 0.6*X_val_images + 0.2
conditions = {"Originale": X_val_images, "Sfocatura 3×3": blurred, "Contrasto 0.6": low_contrast}
shift_results, shift_predictions = {}, {}
for name, batch in conditions.items():
    pred, prob = predict_knn(batch)
    correct = pred == y_val
    shift_predictions[name] = pred
    shift_results[name] = {"n": len(y_val), "correct": int(correct.sum()),
        "accuracy": float(correct.mean()),
        "clean_correct_to_error": int(np.sum((pred_clean == y_val) & ~correct)),
        "clean_error_to_correct": int(np.sum((pred_clean != y_val) & correct))}
table(["Condizione", "Corretti / 360", "Accuracy", "Da corretto a errore", "Da errore a corretto"],
      [[name, f'{r["correct"]} / {r["n"]}', f'{r["accuracy"]:.6f}',
        r["clean_correct_to_error"], r["clean_error_to_correct"]] for name, r in shift_results.items()])
with (OUTPUT/"diagnostica_appaiata.csv").open("w", newline="", encoding="utf-8") as stream:
    writer = csv.writer(stream)
    writer.writerow(["original_index", "assigned_label", "pred_original", "pred_blur", "pred_contrast"])
    writer.writerows(zip(val_idx.tolist(), y_val.tolist(),
                        shift_predictions["Originale"].tolist(), shift_predictions["Sfocatura 3×3"].tolist(),
                        shift_predictions["Contrasto 0.6"].tolist()))

## C08 — Leggere uno stress test

Il numero dei casi passati da corretto a errore e quello dei casi recuperati descrivono il cambiamento sulle **stesse unità**. La differenza netta di accuracy non dice quali immagini siano cambiate, né misura tutte le possibili acquisizioni.

Nel pannello C09 mostriamo i primi sei casi, nell’ordine della validation, in cui la sfocatura cambia la classe prevista. È una selezione di errori da discutere, non un campione casuale rappresentativo della frequenza complessiva.

Per valutare una nuova camera serve un protocollo esterno rappresentativo dell’uso: gruppi indipendenti per sessione/scena, annotazioni controllate e scelte congelate prima della valutazione. Fare molti frame della stessa sequenza non aumenta nello stesso modo il numero di acquisizioni indipendenti.

In [ ]:
# C09 — Due letture complementari dello stesso confronto
changed = np.flatnonzero(shift_predictions["Sfocatura 3×3"] != pred_clean)
chosen = np.r_[changed[:6], np.setdiff1d(np.arange(len(y_val)), changed)[:6]][:6]
fig, axes = plt.subplots(3, 6, figsize=(12, 6))
for row, (name, batch) in enumerate(conditions.items()):
    for col, i in enumerate(chosen):
        ax = axes[row, col]
        ax.imshow(batch[i], cmap="gray", vmin=0, vmax=1, interpolation="nearest")
        ax.set_title(f"etichetta {y_val[i]} · pred {shift_predictions[name][i]}", fontsize=9)
        ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
        if col == 0: ax.set_ylabel(name, fontsize=11)
fig.suptitle("Diagnostica appaiata · Digits validation già vista · kNN k=5", fontsize=14)
fig.tight_layout()
savefig("01_corruzioni_appaiate.png")
fig, ax = plt.subplots(figsize=(8, 4.2))
bars = ax.bar(list(shift_results), [r["accuracy"] for r in shift_results.values()],
              color=[COLORS["teal"], COLORS["orange"], COLORS["blue"]])
ax.set_ylim(0, 1.08); ax.set_ylabel("Accuracy · 360 immagini")
ax.set_title("Misura locale su validation riusata, non nuovo test esterno")
for bar, r in zip(bars, shift_results.values()):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.015,
            f'{r["correct"]}/360 = {r["accuracy"]:.3f}', ha="center")
savefig("02_accuracy_corruzioni.png")

## C10 — Gruppi di acquisizione e numerosità

Un gruppo è una condizione documentata di acquisizione, per esempio camera, sessione o illuminazione. **Non inferiamo attributi personali dalle fotografie.** Il bias discusso qui è una distorsione sistematica di raccolta, annotazione o decisione; è diverso dal parametro `b` del neurone.

L’esempio seguente è **analitico assegnato**, separato da Digits: gruppo A con 86 corretti su 90 e gruppo B con 4 su 10. Una media aggregata pesa i gruppi secondo la loro presenza nel campione. La media semplice delle accuracy dei gruppi assegna loro lo stesso peso: qui la chiamiamo **media fra gruppi**, non balanced accuracy fra classi.

**N03.** Come convivono accuracy totale `0.9` e accuracy del gruppo peggiore `0.4`? Quale supporto accompagna ciascun valore?

In [ ]:
# C11 — Esempio assegnato: supporto, media aggregata e worst-group
group_n = np.array([90, 10])
group_correct = np.array([86, 4])
group_acc = group_correct/group_n
aggregate = float(group_correct.sum()/group_n.sum())
worst = float(group_acc.min())
mean_groups = float(group_acc.mean())
assert np.isclose(aggregate, 0.9) and np.isclose(worst, 0.4)
assert np.isclose(mean_groups, 0.6777777777777778)
table(["Gruppo", "Corretti / n", "Accuracy"],
      [[g, f"{c} / {n}", f"{a:.6f}"] for g, c, n, a in zip(["A", "B"], group_correct, group_n, group_acc)]
      + [["Totale", "90 / 100", f"{aggregate:.6f}"], ["Media fra gruppi", "pesi uguali", f"{mean_groups:.6f}"]])
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
axes[0].bar(["A", "B"], group_n, color=[COLORS["teal"], COLORS["orange"]])
axes[0].set_ylabel("Numero di esempi"); axes[0].set_title("Supporto del campione")
axes[1].bar(["A", "B", "Totale"], [*group_acc, aggregate],
            color=[COLORS["teal"], COLORS["orange"], COLORS["blue"]])
axes[1].set_ylim(0, 1.08); axes[1].set_ylabel("Accuracy")
for i, a in enumerate([*group_acc, aggregate]): axes[1].text(i, a+0.015, f"{a:.3f}", ha="center")
axes[1].set_title("Esempio analitico assegnato")
fig.tight_layout(); savefig("03_gruppi_supporto.png")

## C12 — Cambiare i pesi non significa migliorare il metodo

**N04.** Se nell’uso A e B avessero ciascuno peso `0.5`, quale accuracy attesa otterremmo **assumendo invariate** le due accuracy condizionate al gruppo? Il valore è una ripesatura analitica, non una misura su nuovi dati.

I dieci casi del gruppo B sono un supporto piccolo. Un intervallo richiede ipotesi sul campionamento; frame correlati non sono dieci prove indipendenti per il solo fatto di essere dieci file. Definire prima le unità indipendenti del progetto e il confronto che interessa.

In [ ]:
# C13 — Effetto del mix: confronto analitico, tassi per gruppo congelati
weights_observed = group_n/group_n.sum()
weights_equal = np.array([0.5, 0.5])
equal_mix = float(weights_equal@group_acc)
assert np.isclose(equal_mix, 0.6777777777777778)
fig, ax = plt.subplots(figsize=(8, 4))
mix_scores = [aggregate, equal_mix]
bars = ax.bar(["Mix osservato: A 0.9, B 0.1", "Mix ipotetico: A 0.5, B 0.5"], mix_scores,
              color=[COLORS["teal"], COLORS["orange"]])
ax.set_ylim(0, 1.04); ax.set_ylabel("Accuracy pesata")
ax.set_title("Stessi tassi per gruppo, diverso mix · esempio assegnato")
for b, value in zip(bars, mix_scores): ax.text(b.get_x()+b.get_width()/2, value+0.015, f"{value:.6f}", ha="center")
savefig("04_effetto_mix.png")
print("Worst-group invariato:", worst, "· Nessuna nuova misura di dominio esterno.")

## C14 — Score, probabilità, confidenza ed entropia

Uno **score** `s_j` è un numero reale associato alla classe `j`. La softmax trasforma gli score in probabilità `p_j = exp(s_j) / Σ_k exp(s_k)`. La classe prevista è l’argmax; la confidenza top-label è `max_j p_j`. Un valore alto può accompagnare un errore o un input estraneo all’uso verificato.

L’**entropia predittiva** è `H(p) = −Σ_j p_j ln(p_j)`, con `0 ln(0) = 0`: misura quanto la distribuzione è dispersa. Usiamo logaritmi naturali, quindi l’unità è il **nat**. L’incertezza **aleatoria** riguarda ambiguità o rumore del processo osservativo; quella **epistemica** riguarda conoscenza limitata del modello. L’entropia di una singola previsione non separa automaticamente queste due componenti.

In [ ]:
# C15 — Entropia e confidenza: due distribuzioni assegnate
def entropy(p):
    p = np.asarray(p, dtype=np.float64)
    return float(-np.sum(p[p > 0]*np.log(p[p > 0])))
p_uncertain = np.array([0.5, 0.5])
p_confident = np.array([0.9, 0.1])
H_uniform, H_confident = entropy(p_uncertain), entropy(p_confident)
assert np.isclose(H_uniform, np.log(2))
assert np.isclose(H_confident, 0.3250829733914482)
fig, axes = plt.subplots(1, 2, figsize=(8.8, 3.7))
for ax, p in zip(axes, [p_uncertain, p_confident]):
    ax.bar(["classe 0", "classe 1"], p, color=[COLORS["teal"], COLORS["orange"]])
    ax.set_ylim(0, 1); ax.set_ylabel("Probabilità")
    ax.set_title(f"confidenza {p.max():.1f} · H = {entropy(p):.6f} nat")
fig.suptitle("Esempio analitico: la confidenza non certifica la correttezza")
fig.tight_layout(); savefig("05_entropia.png")

## C16 — Qualità delle probabilità: NLL e Brier

Su `n` esempi e `K` classi usiamo:

- **NLL (negative log-likelihood)**: $-\frac{1}{n}\sum_i \ln p_{i,y_i}$; è la log-verosimiglianza negativa media. Si legge la probabilità della **classe vera**, anche quando non è la classe prevista.
- **Brier multiclass**: $\frac{1}{n}\sum_i\sum_j(p_{ij}-\mathbb{1}[y_i=j])^2$; sommiamo sulle classi, senza dividere per `K`. Il valore è in `[0, 2]`. Per due classi vale due volte il Brier binario scalare.

Sono misure della qualità probabilistica, non soltanto della calibrazione. Un’accuracy uguale non comporta NLL o Brier uguali.

**N05.** A e B predicono sempre la classe 0, corretta in 6 casi su 10. A assegna `(0.9, 0.1)`, B `(0.6, 0.4)`. Quali valori otteniamo? Questo è un esempio assegnato, separato dalle misure Digits.

In [ ]:
# C17 — Stessa accuracy, diverse NLL e Brier: dieci esempi assegnati
y_toy = np.r_[np.zeros(6, dtype=int), np.ones(4, dtype=int)]
def probability_metrics(y, p):
    p = np.asarray(p, dtype=np.float64)
    assert p.shape[0] == len(y) and np.allclose(p.sum(axis=1), 1)
    true_p = p[np.arange(len(y)), y]
    nll = float(-np.log(np.clip(true_p, np.finfo(float).tiny, 1)).mean())
    onehot = np.eye(p.shape[1])[y]
    brier = float(np.square(p-onehot).sum(axis=1).mean())
    return {"accuracy": float((p.argmax(axis=1) == y).mean()), "NLL_nat": nll, "Brier_sum_classes": brier}
calibration_comparison = {}
for name, confidence in [("A", 0.9), ("B", 0.6)]:
    probs = np.tile([confidence, 1-confidence], (len(y_toy), 1))
    calibration_comparison[name] = probability_metrics(y_toy, probs)
assert np.isclose(calibration_comparison["A"]["NLL_nat"], 0.9842503465923135)
assert np.isclose(calibration_comparison["B"]["NLL_nat"], 0.6730116670092565)
assert np.isclose(calibration_comparison["A"]["Brier_sum_classes"], 0.66)
assert np.isclose(calibration_comparison["B"]["Brier_sum_classes"], 0.48)
table(["Modello assegnato", "Accuracy", "NLL (nat)", "Brier (somma classi)"],
      [[name, f'{r["accuracy"]:.3f}', f'{r["NLL_nat"]:.6f}', f'{r["Brier_sum_classes"]:.6f}']
       for name, r in calibration_comparison.items()])
fig, axes = plt.subplots(1, 2, figsize=(8.4, 3.6))
for ax, metric in zip(axes, ["NLL_nat", "Brier_sum_classes"]):
    ax.bar(["A: conf. 0.9", "B: conf. 0.6"], [r[metric] for r in calibration_comparison.values()],
           color=[COLORS["orange"], COLORS["teal"]])
    ax.set_ylabel("NLL (nat)" if metric == "NLL_nat" else "Brier · somma sulle classi")
fig.suptitle("Dieci esempi assegnati · 6 corretti per entrambi")
fig.tight_layout(); savefig("06_nll_brier.png")

## C18 — Diagramma di affidabilità ed ECE: un secondo esempio

**Cambiamo insieme di dati.** Ora ci sono due gruppi di cinque predizioni: confidenze `0.6` e `0.9`, accuracy `0.6` in entrambi. Non sono le dieci predizioni dei modelli A e B di C17.

Un **bin** è un intervallo di confidenza. Usiamo `[0, 0.75)` e `[0.75, 1]`; il bordo `0.75` va nel secondo bin. Il diagramma confronta confidenza media e frequenza di correttezza. L’**ECE (Expected Calibration Error) top-label** è $\sum_m(n_m/n)|acc_m-conf_m|$; i bin vuoti contribuiscono zero e non vengono disegnati. È una stima finita dipendente dai bordi, non una garanzia su ogni gruppo futuro.

**N06.** Quale contributo dà ogni bin? Un’ECE bassa dimostra che il modello riconosce input fuori dominio?

In [ ]:
# C19 — ECE separato: due bin, cinque predizioni per bin
ece_conf = np.r_[np.full(5, 0.6), np.full(5, 0.9)]
ece_correct = np.array([1, 1, 1, 0, 0]*2, dtype=bool)
edges = np.array([0.0, 0.75, 1.0])
bin_id = np.searchsorted(edges[1:-1], ece_conf, side="right")
bins, ece_value = [], 0.0
for b in range(len(edges)-1):
    mask = bin_id == b
    n_b = int(mask.sum())
    if n_b == 0:
        continue
    c_b, a_b = float(ece_conf[mask].mean()), float(ece_correct[mask].mean())
    contribution = n_b/len(ece_conf)*abs(a_b-c_b)
    bins.append({"bin": b, "n": n_b, "mean_confidence": c_b,
                 "accuracy": a_b, "contribution": contribution})
    ece_value += contribution
assert np.isclose(ece_value, 0.15)
table(["Bin", "n", "Confidenza media", "Accuracy", "Contributo ECE"],
      [["[0, 0.75)" if r["bin"] == 0 else "[0.75, 1]", r["n"],
        f'{r["mean_confidence"]:.3f}', f'{r["accuracy"]:.3f}', f'{r["contribution"]:.3f}'] for r in bins])
print(f"ECE top-label = {ece_value:.3f} · esempio diverso da C17")
fig, ax = plt.subplots(figsize=(6.5, 4.4))
ax.plot([0, 1], [0, 1], "--", color="0.45", label="accordo ideale")
for r in bins:
    ax.vlines(r["mean_confidence"], r["accuracy"], r["mean_confidence"], color=COLORS["orange"], linewidth=3)
    ax.scatter(r["mean_confidence"], r["accuracy"], s=110, color=COLORS["teal"], zorder=3)
    ax.annotate(f'n = {r["n"]}', (r["mean_confidence"], r["accuracy"]), xytext=(-12, -22), textcoords="offset points")
ax.set(xlim=(0, 1), ylim=(0, 1), xlabel="Confidenza media nel bin", ylabel="Frequenza di correttezza",
       title="Affidabilità top-label · esempio assegnato · ECE = 0.15")
ax.legend(); savefig("07_affidabilita_ece.png")

## C20 — Temperature scaling: trasformazione e selezione sono distinte

$p_j(T)=\exp(s_j/T)/\sum_k\exp(s_k/T)$ con `T > 0`. La **temperatura** cambia quanto la distribuzione è concentrata, senza riaddestrare i pesi. Si sceglie `T` su un insieme di calibrazione separato dalla valutazione finale, spesso minimizzando la NLL. Non si sceglie sul test.

Qui `T = 2` è **assegnato**, non stimato dai dati. L’argmax degli score di un singolo esempio resta invariato; ciò non garantisce che il ranking delle confidenze fra esempi multiclass resti uguale o che ogni ECE migliori.

**N07.** Con score `s = (ln(4), 0)`, quali probabilità otteniamo per `T = 1` e `T = 2`? Che cosa succede alla classe prevista?

In [ ]:
# C21 — Temperatura assegnata: esempio analitico verificato
def softmax(s):
    s = np.asarray(s, dtype=np.float64)
    exp_s = np.exp(s-s.max(axis=-1, keepdims=True))
    return exp_s/exp_s.sum(axis=-1, keepdims=True)
scores = np.array([np.log(4), 0.0])
before, after = softmax(scores), softmax(scores/2)
assert np.allclose(before, [0.8, 0.2])
assert np.allclose(after, [2/3, 1/3])
assert before.argmax() == after.argmax() == 0
table(["T", "p(classe 0)", "p(classe 1)", "Classe prevista"],
      [[1, f"{before[0]:.6f}", f"{before[1]:.6f}", 0], [2, f"{after[0]:.6f}", f"{after[1]:.6f}", 0]])
fig, ax = plt.subplots(figsize=(7.5, 3.8))
x = np.arange(2); width = 0.34
ax.bar(x-width/2, before, width, label="T = 1", color=COLORS["orange"])
ax.bar(x+width/2, after, width, label="T = 2 assegnato", color=COLORS["teal"])
ax.set_xticks(x, ["classe 0", "classe 1"])
ax.set_ylim(0, 1); ax.set_ylabel("Probabilità")
ax.set_title("Score s = (ln(4), 0) · argmax invariato")
ax.legend(); savefig("08_temperatura.png")

## C22 — Astensione, copertura e rischio selettivo

Accettiamo una previsione se la confidenza è **maggiore o uguale** alla soglia `τ`. La **copertura** è `accettati / totale`; il **rischio selettivo** è `errori fra gli accettati / accettati`. Senza accettati il rischio non è definito: non è zero. L’astensione richiede un’alternativa esplicita, per esempio revisione umana o nuova acquisizione; il caso non scompare dal sistema.

**N08.** Negli otto casi assegnati, che cosa accade per `τ = 0.9` e `τ = 0.8`? Perché riportare anche la copertura di ciascun gruppo? La curva empirica può avere tratti in cui il rischio diminuisce all’aumentare della copertura; non imponiamo monotonicità al grafico.

In un progetto la soglia va scelta su calibrazione/validation e poi congelata. Questi otto casi illustrano il calcolo, senza certificare una soglia per un dominio futuro.

In [ ]:
# C23 — Rischio/copertura: otto casi analitici, nessuna soglia ottimizzata sul test
sel_conf = np.array([0.99, 0.95, 0.9, 0.85, 0.8, 0.75, 0.7, 0.6])
sel_error = np.array([0, 0, 0, 1, 0, 1, 1, 1])
def selective_metrics(threshold):
    accepted = sel_conf >= threshold
    n_a = int(accepted.sum())
    return {"threshold": float(threshold), "accepted": n_a,
            "errors": int(sel_error[accepted].sum()), "coverage": n_a/len(sel_conf),
            "risk": float(sel_error[accepted].mean()) if n_a else None}
selective_rows = [selective_metrics(t) for t in [1.0, *sel_conf]]
r09, r08, rall = selective_metrics(0.9), selective_metrics(0.8), selective_metrics(0.0)
assert r09 == {"threshold": 0.9, "accepted": 3, "errors": 0, "coverage": 0.375, "risk": 0.0}
assert r08 == {"threshold": 0.8, "accepted": 5, "errors": 1, "coverage": 0.625, "risk": 0.2}
assert rall["coverage"] == 1 and rall["risk"] == 0.5
assert selective_metrics(1.0)["risk"] is None
table(["Soglia τ", "Accettati", "Errori", "Copertura", "Rischio"],
      [[f'{r["threshold"]:.2f}', r["accepted"], r["errors"], f'{r["coverage"]:.3f}',
        "non definito" if r["risk"] is None else f'{r["risk"]:.3f}'] for r in selective_rows])
fig, ax = plt.subplots(figsize=(7.5, 4.2))
valid = [r for r in selective_rows if r["risk"] is not None]
ax.plot([r["coverage"] for r in valid], [r["risk"] for r in valid], "o-", color=COLORS["teal"])
for r in [r09, r08, rall]:
    ax.annotate(f'τ={r["threshold"]:.1f}', (r["coverage"], r["risk"]), xytext=(4, 8), textcoords="offset points")
ax.set(xlim=(0, 1.03), ylim=(-0.03, 0.58), xlabel="Copertura · accettati / 8",
       ylabel="Rischio · errori / accettati", title="Esempio assegnato · rischio a copertura zero non definito")
savefig("09_rischio_copertura.png")

## C24 — Un benchmark prima dei numeri

La **latenza** è il tempo impiegato per una richiesta; il **throughput** è quante richieste vengono elaborate per unità di tempo. Qui misuriamo sul computer che esegue il notebook: **CPU, batch 1, 64 valori float64, kNN k=5, 1077 riferimenti**.

- **Predizione:** l’input è già normalizzato e la baseline è già in memoria.
- **Preprocessing + predizione:** partiamo dalla stessa immagine `uint8`, convertiamo a `float64`, dividiamo per 16 e prediciamo. Escludiamo lettura del file, visualizzazione e avvio del processo; non lo chiamiamo tempo completo dell’applicazione.

Usiamo 5 richieste di riscaldamento, poi 30 richieste temporizzate per ciascuna modalità. `p50` è la mediana e `p95` il 95° percentile, con interpolazione lineare NumPy. Le trenta richieste **non** sono trenta addestramenti o repliche indipendenti della qualità del modello.

**N09.** Perché non confrontare direttamente questi tempi con una GPU, un altro batch o un’app che legge le immagini da disco?

In [ ]:
# C25 — Misura locale, senza tempi precostituiti
query_uint8 = images[val_idx[0]].copy()
query_normalized = query_uint8.astype(np.float64)/16.0
WARMUP, REPETITIONS = 5, 30
timing_functions = {"predizione": lambda: predict_knn(query_normalized),
                    "preprocessing_e_predizione": lambda: predict_knn(query_uint8.astype(np.float64)/16.0)}
timing_samples_ms, timing_summary = {}, {}
for name, function in timing_functions.items():
    for _ in range(WARMUP):
        function()
    samples = []
    for _ in range(REPETITIONS):
        start = time.perf_counter_ns()
        result = function()
        elapsed = (time.perf_counter_ns()-start)/1e6
        samples.append(elapsed)
    arr = np.array(samples)
    timing_samples_ms[name] = samples
    timing_summary[name] = {"p50_ms": float(np.percentile(arr, 50, method="linear")),
        "p95_ms": float(np.percentile(arr, 95, method="linear")), "mean_ms": float(arr.mean()),
        "sequential_requests_per_second": float(1000/arr.mean()), "warmup": WARMUP,
        "repetitions": REPETITIONS, "batch_size": 1, "dtype": "float64", "device": "CPU"}
    assert timing_summary[name]["p95_ms"] >= timing_summary[name]["p50_ms"] > 0
table(["Modalità", "p50 ms", "p95 ms", "Media ms", "Richieste/s sequenziali"],
      [[name, f'{r["p50_ms"]:.4f}', f'{r["p95_ms"]:.4f}', f'{r["mean_ms"]:.4f}',
        f'{r["sequential_requests_per_second"]:.1f}'] for name, r in timing_summary.items()])
print("Misure del runtime corrente:", "Colab CPU" if IN_COLAB else "locale CPU")

## C26 — Efficienza, variabilità e compromesso

Il `p95` di questa corsa descrive la coda dei tempi osservati nel protocollo; non garantisce un limite futuro. Ripetere in un ambiente libero da altri carichi può cambiare il risultato. La versione della libreria e il backend numerico fanno parte del contesto.

Su un compito così breve, cache e variabilità possono anche invertire l’ordine dei tempi delle due modalità: non concludiamo che aggiungere preprocessing acceleri la predizione. Un confronto robusto richiederebbe più sessioni con protocollo controllato.

Due modelli possono richiedere confronti su più assi. Un punto è **dominato** se un altro ha accuracy non inferiore e latenza non maggiore, con almeno una disuguaglianza stretta. La **frontiera di Pareto** raccoglie i punti non dominati. Nel secondo grafico seguente i numeri sono **assegnati**, separati dal benchmark kNN appena misurato.

In [ ]:
# C27 — Tempi osservati e Pareto assegnato: due figure, due fonti di evidenza
fig, ax = plt.subplots(figsize=(9, 4))
for i, (name, samples) in enumerate(timing_samples_ms.items()):
    ax.scatter(samples, np.full(len(samples), i), alpha=0.45, s=28, color=[COLORS["teal"], COLORS["orange"]][i])
    r = timing_summary[name]
    ax.plot([r["p50_ms"], r["p95_ms"]], [i+0.12, i+0.12], "s-", color=COLORS["blue"])
    ax.annotate(f'p50 {r["p50_ms"]:.3f} · p95 {r["p95_ms"]:.3f} ms',
                (r["p50_ms"], i+0.12), xytext=(4, 6), textcoords="offset points", fontsize=9)
ax.set_yticks([0, 1], ["Predizione", "Preprocessing + predizione"])
ax.set_ylim(-0.3, 1.5); ax.set_xlabel("Latenza osservata (ms)")
ax.set_title("kNN · CPU · batch 1 · 5 warmup + 30 richieste per modalità")
savefig("10_benchmark_locale.png")
pareto_names = np.array(["M1", "M2", "M3", "M4", "M5"])
pareto_accuracy = np.array([0.88, 0.9, 0.89, 0.87, 0.91])
pareto_latency = np.array([12, 20, 25, 18, 40])
dominated = np.array([any((pareto_accuracy[j] >= pareto_accuracy[i]) and
    (pareto_latency[j] <= pareto_latency[i]) and
    ((pareto_accuracy[j] > pareto_accuracy[i]) or (pareto_latency[j] < pareto_latency[i]))
    for j in range(5) if j != i) for i in range(5)])
assert pareto_names[~dominated].tolist() == ["M1", "M2", "M5"]
fig, ax = plt.subplots(figsize=(7.6, 4.2))
ax.scatter(pareto_latency[dominated], pareto_accuracy[dominated], color=COLORS["orange"], s=90, label="dominati")
ax.plot(pareto_latency[~dominated], pareto_accuracy[~dominated], "o-", color=COLORS["teal"], label="frontiera")
for name, x, y in zip(pareto_names, pareto_latency, pareto_accuracy): ax.annotate(name, (x, y), xytext=(5, 5), textcoords="offset points")
ax.set(xlabel="Latenza assegnata (ms) · minimizzare", ylabel="Accuracy assegnata · massimizzare",
       ylim=(0.86, 0.92), title="Esempio analitico: nessuna misura hardware")
ax.legend(); savefig("11_pareto_assegnato.png")

## C28 — Riproducibilità: che cosa conserviamo?

Il **manifesto** è un riepilogo strutturato che collega risultati, dati, indici, trasformazioni, modello, ambiente e protocollo. Un hash SHA-256 identifica i byte del file; non ne dimostra correttezza, liceità o assenza di quasi duplicati. Il seed inizializza un generatore casuale, ma non garantisce uguaglianza fra hardware, versioni e implementazioni.

**N10.** Che cosa può verificare una persona che riapre questi materiali? Quale affermazione resta non verificata? Esportare un manifesto non equivale a eseguire una prova da un ambiente nuovo.

In [ ]:
# C29 — Manifesto, risultati e protocollo leggibili separatamente
environment = {"python": platform.python_version(), "numpy": np.__version__,
    "matplotlib": matplotlib.__version__, "system": platform.system(),
    "release": platform.release(), "machine": platform.machine(),
    "runtime": "Colab CPU" if IN_COLAB else "locale CPU", "device": "CPU",
    "thread_environment": {k: os.environ.get(k, "non specificato")
        for k in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"]}}
manifest = {"course": "Computer Vision", "lesson": 21, "protocol_version": "L21_20261130_v1",
    "seed": SEED, "environment": environment, "dataset_sha256": DATA_SHA256,
    "dataset": "Digits UCI, copia scikit-learn distribuita nel corso",
    "split": {"train_n": len(train_idx), "validation_reused_n": len(val_idx),
              "train_indices": train_idx.tolist(), "validation_indices": val_idx.tolist(),
              "test_evaluated": False, "writer_disjointness_verified": False},
    "model": {"method": "kNN", "k": K_NEIGHBORS, "distance": "Euclidea su pixel /16",
              "tie_rule": "classe con indice minore; vicini ordinati stabilmente",
              "vote_smoothing_per_class": VOTE_SMOOTHING, "confidence_calibrated": False},
    "transforms": {"blur": "media 3x3, bordo edge", "contrast": "0.6*x+0.2", "paired": True},
    "scope": "Diagnostica appaiata su validation già vista; non nuovo test esterno",
    "timing_protocol": {"warmup": WARMUP, "requests_per_mode": REPETITIONS, "batch": 1,
                        "input_shape": [1, 64], "dtype": "float64", "percentile_method": "linear",
                        "excluded": ["I/O file", "avvio processo", "visualizzazione"],
                        "not_independent_training_replicates": True},
    "new_environment_reproduction_verified": False}
results = {"measured_validation_diagnostics": shift_results, "measured_current_runtime_cpu_timing": timing_summary,
    "assigned_analytic_examples": {"groups": {"aggregate": aggregate, "worst": worst,
      "mean_groups": mean_groups, "equal_mix": equal_mix}, "entropy_nat": [H_uniform, H_confident],
      "probability_comparison": calibration_comparison, "ECE_separate_example": {"bins": bins, "value": ece_value},
      "temperature": {"T": 2, "before": before.tolist(), "after": after.tolist(), "fitted": False},
      "selection": selective_rows, "Pareto_frontier": pareto_names[~dominated].tolist()}}
for filename, obj in [("manifesto.json", manifest), ("risultati.json", results), ("tempi_grezzi_ms.json", timing_samples_ms)]:
    (OUTPUT/filename).write_text(json.dumps(obj, indent=2, ensure_ascii=False, allow_nan=False)+"\n", encoding="utf-8")
with (OUTPUT/"rischio_copertura.csv").open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=list(selective_rows[0]))
    writer.writeheader(); writer.writerows(selective_rows)
print("Salvati manifesto, risultati, tempi grezzi e CSV; figure salvate durante i blocchi precedenti.")

## C30 — Dalla dimostrazione alla clinic

Per il progetto aprire le tre schede in `consegna/`:

- **Data card:** raccolta, annotazioni, unità indipendente, split, licenza e limiti.
- **Model card:** uso previsto, dati/versioni, metriche, efficienza, usi non verificati e alternativa in caso di astensione.
- **Clinic:** una sola affermazione principale, un’evidenza, un limite e una prossima prova compatibile con il budget.

Non trasferire automaticamente i numeri assegnati o le prestazioni Digits al proprio progetto. La clinic usa il lavoro già svolto: non aggiunge un assignment, un nuovo benchmark obbligatorio o un addestramento lungo. Rimangono i vincoli del catalogo dei progetti, inclusi i gruppi individuali o da 2–3 persone e il massimo di 120 ore per persona.

In [ ]:
# C31 — Archivio degli output e controllo finale
exported = sorted(p for p in OUTPUT.iterdir() if p.is_file() and p.suffix in {".png", ".json", ".csv"}
                  and p.name != "elenco_file.json")
checksum_rows = [{"file": p.name, "sha256": hashlib.sha256(p.read_bytes()).hexdigest(),
                  "bytes": p.stat().st_size} for p in exported]
(OUTPUT/"elenco_file.json").write_text(json.dumps(checksum_rows, indent=2, ensure_ascii=False)+"\n", encoding="utf-8")
archive_path = OUTPUT.parent/"output_lezione21.zip"
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in [*exported, OUTPUT/"elenco_file.json"]:
        archive.write(path, arcname=path.name)
print(f"Archivio pronto: {archive_path.name} · {len(exported)+1} file")
print("Controlli numerici analitici superati. Il notebook non valuta il test e non certifica un dominio futuro.")
display(FileLink(archive_path.name))
if IN_COLAB and DOWNLOAD_OUTPUTS:
    from google.colab import files
    files.download(str(archive_path))

## C32 — Fonti e letture

- E. Alpaydin e C. Kaynak, **Optical Recognition of Handwritten Digits**, UCI, 1998. [Dataset e licenza](https://doi.org/10.24432/C50P49); [load_digits](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html). Copia dei dati e attribuzione in `dati/README.md`.
- D. Hendrycks e T. Dietterich, **Benchmarking Neural Network Robustness to Common Corruptions and Perturbations**, ICLR 2019. [Articolo](https://arxiv.org/abs/1903.12261). La nostra sfocatura didattica non riproduce quel benchmark.
- C. Guo et al., **On Calibration of Modern Neural Networks**, ICML 2017. [Articolo PMLR](https://proceedings.mlr.press/v70/guo17a.html).
- Y. Geifman e R. El-Yaniv, **Selective Classification for Deep Neural Networks**, NeurIPS 2017. [Articolo](https://arxiv.org/abs/1705.08500).
- M. Mitchell et al., **Model Cards for Model Reporting**, FAccT 2019. [Articolo](https://arxiv.org/abs/1810.03993).
- T. Gebru et al., **Datasheets for Datasets**, CACM 2021. [Articolo](https://arxiv.org/abs/1803.09010).
- M. Pushkarna et al., **Data Cards: Purposeful and Transparent Dataset Documentation for Responsible AI**, FAccT 2022. [Articolo](https://arxiv.org/abs/2204.01075).